In [19]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import SystemMessage,HumanMessage,BaseMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [20]:
load_dotenv()

True

In [21]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [22]:
llm=ChatGroq(model='llama-3.1-8b-instant')
def chat_node(state:ChatState):
    messages=state['messages']
    response=llm.invoke(messages)
    return {'messages':[response]}

In [26]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpointer)

In [27]:
thread_id='1'
while True:
    user_message=input('Type Here')
    print('User:',user_message)
    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    config={'configurable':{'thread_id':thread_id}}
    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    print('AI:',response['messages'][-1].content)

User: hi my name s khushi
AI: Nice to meet you, Khushi! How are you doing today? Is there anything I can help you with or would you like to chat?
User: can you tell me my name
AI: Your name is Khushi.
User: i m 21 years old
AI: You're 21 years old, that's a great age. You're likely in a stage of self-discovery and figuring out your career and personal life. What are your interests or hobbies, Khushi?
User: i love ai
AI: That's fascinating. You're interested in Artificial Intelligence, or AI. AI is a rapidly growing field with many applications in technology, healthcare, finance, and more. What specifically about AI interests you, Khushi? Do you want to learn more about machine learning, natural language processing, or something else?
User: tell me about me
AI: Based on our conversation, here's what I know about you, Khushi:

1. **Age**: You're 21 years old, which is a great age for exploring your interests and passions.
2. **Name**: Your name is Khushi, which is a beautiful and cheerfu